In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm
tqdm.pandas()

from sklearn.metrics import classification_report

from pydantic import BaseModel, Field
from enum import Enum

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

from dotenv import load_dotenv
load_dotenv()

True

### 1. 예상 질의
* 예상 질의는 90개로 구성
* 3개의 intent로 구분
    1. 복합 질의
        * 다단계의 논리적 처리 또는 다중 도구 호출이 필요한 질의(40개)
            * RAG + Text to SQL
            * 복잡한 Text to SQL
        * 하나의 도구 호출로 완료될 수 없으며, 중간 결과의 통합, 비교, 추론을 위해 plan and excute 구조 기반 답변
        * 예시
            * 나의 용상 동작과 인상 동작에서 공통적으로 발생하는 고질적인 문제점이 무엇이야?
            * 내가 올린 영상 중 평균 점수 80점 미만인 영상들의 가장 흔한 오류 부위는 무엇이야?
            * 내가 올린 영상 중 평균 점수 95점 이상인 영상들의 공통적인 특징을 분석하고, 기술 구조를 참조하여 요약해줘.
    2. 단순 질의
        * 하나의 도구 호출이 필요한 질의(30개)
            * 단순 RAG
            * 단순 Text to SQL
        * 예시
            * 지면 반발력(Ground Reaction Force)이 역도 동작에 어떻게 활용되나요?
            * 스내치(Snatch) 동작에서 바벨을 받는 시점에 대한 기술적 조언을 해주세요.
            * 가장 최근에 올린 영상의 전체 점수는 몇점이야?
    3. 예외 질의
        * 서비스의 범위를 벗어나거나(일반 대화, 날씨 등), 현재 지원되지 않는 기능(영상 편집, 계정 관리 등)에 대한 질의(20개)
        * 기존 저장된 메시지로 답변 출력력
        * 예시
            * 파이썬으로 DTW 알고리즘을 구현하는 코드를 알려주세요.
            * 가장 가까운 역도 체육관 위치를 찾아주세요.
            * 영상 ID 500을 삭제해 주세요.

In [11]:
query_path = '..\\data\\rag\\intent_queries_90.csv'

query_df = pd.read_csv(query_path)

In [5]:
test_df = query_df.groupby('category').sample(frac=0.2)
query_df['type'] = np.nan
query_df.loc[~query_df.index.isin(test_df.index), 'type'] = 'validation'
query_df.loc[query_df.index.isin(test_df.index), 'type'] = 'test'

# query_df.to_csv(query_path, encoding='utf-8-sig', index=False)

C:\Users\owner\AppData\Local\Temp\ipykernel_25824\4281981954.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'validation' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  query_df.loc[~query_df.index.isin(test_df.index), 'type'] = 'validation'


### 2. Intent Analyze
#### 개요
* 사용자 쿼리의 intent(의도) 분석 및 query rewrite 작업
* intent 종류:
    1. COMPLEX: 다단계 처리 또는 멀티툴 사용 요구 질의 (plan & execute sub-graph 처리)
    2. SIMPLE: 단일 응답으로 해결 가능한 간단 질의
    3. INAPPROPRIATE: 서비스 범위 외 또는 부적절 질의

#### 실험 결과
* intent 분석용 프롬프트 기법 비교(COT, COD)

* COT 기법 성능
    * 전체 정확도: 80.0%
    * 전체 Precision: 83.53%, Recall: 80.00%, F1-Score: 76.22%
    * COMPLEX: Precision 81.63%, Recall 100.00%, F1-Score 89.89%
    * SIMPLE: Precision 100.00%, Recall 40.00%, F1-Score 57.14%
    * INAPPROPRIATE: Precision 68.97%, Recall 100.00%, F1-Score 81.63%
    * 특징: COMPLEX와 INAPPROPRIATE는 Recall 100%로 완벽히 포착하나, Precision이 낮아 false positive 발생
    * SIMPLE의 Recall 40%로 SIMPLE 쿼리의 대부분을 COMPLEX로 오분류하는 경향

* COD 기법 성능
    * 전체 정확도: 80.0%
    * 전체 Precision: 86.80%, Recall: 73.06%, F1-Score: 72.45%
    * COMPLEX: Precision 97.37%, Recall 92.50%, F1-Score 94.87%
    * SIMPLE: Precision 63.04%, Recall 96.67%, F1-Score 76.32%
    * INAPPROPRIATE: Precision 100.00%, Recall 30.00%, F1-Score 46.15%
    * 특징: COMPLEX와 SIMPLE에서 균형잡힌 성능, INAPPROPRIATE의 Recall 30%로 오분류 위험

* 분석 및 결론
    * 두 기법 모두 전체 정확도 80.0%로 동일
    * COT: INAPPROPRIATE Recall 100%로 부적절 쿼리를 확실히 차단하지만, SIMPLE 쿼리의 대부분을 COMPLEX로 오분류
    * COD: COMPLEX와 SIMPLE에서 더 균형잡힌 성능을 보이지만, INAPPROPRIATE Recall 30%로 부적절 쿼리를 놓칠 위험 높음
    * 서비스 관점: INAPPROPRIATE 분류 실패는 사용자 경험에 직접적 영향 → COT 기법 선택 권장
    * COT의 SIMPLE→COMPLEX 오분류는 서비스 관점에서 큰 문제 아님 (더 많은 처리 리소스 사용하지만 결과는 제공)

In [2]:
llm = ChatOpenAI(
    temperature=0.2,
    model="gpt-4o-mini",
)

In [ ]:
class Category(str, Enum):
    """Defines the query processing categories."""
    COMPLEX = "COMPLEX"
    SIMPLE = "SIMPLE"
    INAPPROPRIATE = "INAPPROPRIATE"

class Intent(BaseModel):
    """Schema containing the user query’s intent, processing category, and rewritten query."""
    category: Category = Field(
        description="Query processing category. Must be one of 'COMPLEX', 'SIMPLE', or 'INAPPROPRIATE'."
    )
    query_rewrite: str = Field(
        description="A rewritten version of the original query to make it easier for downstream Agents to process. If the category is INAPPROPRIATE, contains a rejection message."
    )

parser = PydanticOutputParser(pydantic_object=Intent)

#### 2-01. COT

In [ ]:
# from langchain.prompts import PromptTemplate

intent_template = """
[SYSTEM ROLE]
You are the Intent Router for a weightlifting analysis service. Your task is to analyze the user's query and determine the appropriate processing Category and a Rewritten Query for the next processing step.

[CATEGORY DEFINITIONS]
1. COMPLEX: Requires multi-step processing, multi-tool usage (SQL + RAG/LLM Inference), or logical comparison/analysis (Plan-and-Execute).
2. SIMPLE: Requires a single tool call (RAG or Single SQL) (Execute Only).
3. INAPPROPRIATE: Outside the service scope (OUT_OF_SCOPE) or unsupported feature (UNHANDLED_TOOL).

[OUTPUT FORMAT]
{format}

[FEW-SHOT EXAMPLE]

User Query: "What are the common chronic faults in my Snatch and Clean & Jerk movements?"

Thought:
1. Analysis: The user is asking for a comparison and synthesis of data from two different lift types (Snatch and C&J).
2. Tooling: This requires multiple SQL queries (Snatch data, C&J data), followed by LLM inference to find commonalities, and finally RAG for technical explanation.
3. Conclusion: This is a multi-step process requiring planning. The Category is COMPLEX.

Output:
{{
  "category": "COMPLEX",
  "query_rewrite": "Calculate the overall average DTW score of all user videos and compare it with the DTW score of the most recently uploaded video."
}}

[USER QUERY]
{user_query}
"""

In [ ]:
prompt = PromptTemplate.from_template(template=intent_template)
prompt = prompt.partial(format=parser.get_format_instructions())
cot_chain = prompt | llm

In [ ]:
query_df['cot_category'] = np.nan
query_df['cot_rewrite'] = np.nan

cot_category_list = []
cot_rewrite_list = []

for i, row in tqdm(query_df.iterrows()):
    # result = cot_chain.invoke({'user_query': row['query']})
    structed_output = parser.parse(result.content)
    cot_category_list.append(structed_output.category.value)
    cot_rewrite_list.append(structed_output.query_rewrite)

query_df['cot_category'] = cot_category_list
query_df['cot_rewrite'] = cot_rewrite_list

# query_df.to_csv('..\\data\\rag\\intent_queries_90.csv', encoding='utf-8-sig', index=False)

In [65]:
query_df.head(2)

,category,query,type,cot_category,cot_rewrite
0,COMPLEX,어제 올린 용상 영상과 저번 주에 올린 영상 중 가장 잘했던 영상의 DTW 분석 결...,validation,COMPLEX,Compare the DTW analysis results of the best v...
1,COMPLEX,나의 용상 동작과 인상 동작에서 공통적으로 발생하는 고질적인 문제점이 무엇이야?,validation,COMPLEX,나의 용상 동작과 인상 동작에서 공통적으로 발생하는 고질적인 문제점을 분석해줘.


In [14]:
# 전체 정확도
cot_accuracy = round(len(query_df.loc[(query_df['category'] == query_df['cot_category'])]) / len(query_df) * 100, 2)
print(f"COT 전체 정확도: {cot_accuracy}%")

# 분류 보고서 (Precision, Recall, F1-Score 포함)
cot_report = classification_report(
    query_df['category'], 
    query_df['cot_category'],
    target_names=['COMPLEX', 'INAPPROPRIATE', 'SIMPLE'],
    output_dict=True
)

print("\n=== COT 분류 보고서 ===")
print(f"전체 Precision: {cot_report['macro avg']['precision']:.2%}")
print(f"전체 Recall: {cot_report['macro avg']['recall']:.2%}")
print(f"전체 F1-Score: {cot_report['macro avg']['f1-score']:.2%}")

print("\n=== 카테고리별 성능 ===")
for category in ['COMPLEX', 'INAPPROPRIATE', 'SIMPLE']:
    if category in cot_report:
        print(f"\n{category}:")
        print(f"  Precision: {cot_report[category]['precision']:.2%}")
        print(f"  Recall: {cot_report[category]['recall']:.2%}")
        print(f"  F1-Score: {cot_report[category]['f1-score']:.2%}")
        print(f"  Support: {cot_report[category]['support']}")

COT 전체 정확도: 80.0%

=== COT 분류 보고서 ===
전체 Precision: 83.53%
전체 Recall: 80.00%
전체 F1-Score: 76.22%

=== 카테고리별 성능 ===

COMPLEX:
  Precision: 81.63%
  Recall: 100.00%
  F1-Score: 89.89%
  Support: 40.0

INAPPROPRIATE:
  Precision: 68.97%
  Recall: 100.00%
  F1-Score: 81.63%
  Support: 20.0

SIMPLE:
  Precision: 100.00%
  Recall: 40.00%
  F1-Score: 57.14%
  Support: 30.0


#### 2-02. COD

In [ ]:
intent_template = """
[SYSTEM ROLE]
You are the Intent Router for a weightlifting analysis service. Your task is to analyze the user's query and determine the appropriate processing Category and a Rewritten Query for the next processing step.

[CATEGORY DEFINITIONS]
1. COMPLEX: Requires multi-step processing, multi-tool usage (SQL + RAG/LLM Inference), or logical comparison/analysis (Plan-and-Execute).
2. SIMPLE: Requires a single tool call (RAG or Single SQL) (Execute Only).
3. INAPPROPRIATE: Outside the service scope (OUT_OF_SCOPE) or unsupported feature (UNHANDLED_TOOL).


[OUTPUT FORMAT]
{format}

[FEW-SHOT EXAMPLE]
User Query: "Compare Snatch and Clean & Jerk techniques to find common faults."

Draft:
- Requires comparison of multiple datasets.
- Multi-step reasoning → COMPLEX.

Output:
{{
  "category": "COMPLEX",
  "query_rewrite": "Identify and compare technical faults between Snatch and Clean & Jerk movements."
}}

User Query: "Why should the bar stay close to the body during a Snatch pull?"

Draft:
- Knowledge question.
- Single RAG retrieval → SIMPLE.

Output:
{{
  "category": "SIMPLE",
  "query_rewrite": "Explain the technical reason for keeping the bar close to the body during the Snatch pull."
}}

[USER QUERY]
{user_query}
"""

In [73]:
prompt = PromptTemplate.from_template(template=intent_template)
prompt = prompt.partial(format=parser.get_format_instructions())
cod_chain = prompt | llm

In [ ]:
query_df['cod_category'] = np.nan
query_df['cod_rewrite'] = np.nan

cod_category_list = []
cod_rewrite_list = []

for i, row in tqdm(query_df.iterrows()):
    # result = cod_chain.invoke({'user_query': row['query']})
    structed_output = parser.parse(result.content)
    cod_category_list.append(structed_output.category.value)
    cod_rewrite_list.append(structed_output.query_rewrite)

query_df['cod_category'] = cod_category_list
query_df['cod_rewrite'] = cod_rewrite_list

# query_df.to_csv('..\\data\\rag\\intent_queries_90.csv', encoding='utf-8-sig', index=False)

90it [04:01,  2.68s/it]


In [15]:
cod_accuracy = round(len(query_df.loc[(query_df['category'] == query_df['cod_category'])]) / len(query_df) * 100, 2)
print(f"COD 전체 정확도: {cod_accuracy}%")

# 분류 보고서 (Precision, Recall, F1-Score 포함)
cod_report = classification_report(
    query_df['category'], 
    query_df['cod_category'],
    target_names=['COMPLEX', 'INAPPROPRIATE', 'SIMPLE'],
    output_dict=True
)

print("\n=== COD 분류 보고서 ===")
print(f"전체 Precision: {cod_report['macro avg']['precision']:.2%}")
print(f"전체 Recall: {cod_report['macro avg']['recall']:.2%}")
print(f"전체 F1-Score: {cod_report['macro avg']['f1-score']:.2%}")

print("\n=== 카테고리별 성능 ===")
for category in ['COMPLEX', 'INAPPROPRIATE', 'SIMPLE']:
    if category in cod_report:
        print(f"\n{category}:")
        print(f"  Precision: {cod_report[category]['precision']:.2%}")
        print(f"  Recall: {cod_report[category]['recall']:.2%}")
        print(f"  F1-Score: {cod_report[category]['f1-score']:.2%}")
        print(f"  Support: {cod_report[category]['support']}")

COD 전체 정확도: 80.0%

=== COD 분류 보고서 ===
전체 Precision: 86.80%
전체 Recall: 73.06%
전체 F1-Score: 72.45%

=== 카테고리별 성능 ===

COMPLEX:
  Precision: 97.37%
  Recall: 92.50%
  F1-Score: 94.87%
  Support: 40.0

INAPPROPRIATE:
  Precision: 100.00%
  Recall: 30.00%
  F1-Score: 46.15%
  Support: 20.0

SIMPLE:
  Precision: 63.04%
  Recall: 96.67%
  F1-Score: 76.32%
  Support: 30.0
